# Colab Setup
Run this notebook top to bottom every session. It restores your work from Drive if a backup exists (so a runtime disconnect doesn't cost you the ~26 min preprocessing run again), otherwise downloads and processes fresh.

In [1]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2. Clone the repo
%cd /content
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

/content
fatal: destination path 'Review-Summarization-LLM' already exists and is not an empty directory.
/content/Review-Summarization-LLM


In [3]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [4]:
# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

CUDA available: True
Device: Tesla T4


In [5]:
# 5. Try restoring raw data + processed data from Drive backup first.
# If no backup exists yet, this just prints a message and does nothing --
# proceed to cells 6-7 to download and process from scratch.
import os

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
restored = False

if os.path.exists(f'{BACKUP_DIR}/clean_reviews.csv'):
    print('Found backup -- restoring processed data...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null
    restored = True
    print('Restored from Drive backup.')
else:
    print('No Drive backup found yet -- run the download + preprocessing cells below.')

Found backup -- restoring processed data...
Restored from Drive backup.


## Only run cells 6-8 if cell 5 printed "No Drive backup found" -- otherwise skip to cell 9

In [6]:
from getpass import getpass
import os

token = getpass("Paste your Kaggle API token: ").strip()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token)

!python data/download_dataset.py

Paste your Kaggle API token: ··········
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.9MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [7]:
# 6. Set up Kaggle API and download dataset (skip if restored above)
# Upload your kaggle.json via the file browser on the left first, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.2MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [8]:
# 7. Run preprocessing (skip if restored above -- this takes ~20-30 min on the full dataset)
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

2026-07-19 18:47:57,960 [INFO] Loaded 568454 rows from data/raw/Reviews.csv
2026-07-19 18:47:59,184 [INFO] drop_missing_and_nulls: 568454 -> 568454 rows
2026-07-19 18:48:00,616 [INFO] remove_duplicates: 568454 -> 567554 rows
2026-07-19 18:48:29,586 [INFO] filter_uninformative (min_words=5): 567554 -> 567534 rows
object address  : 0x7d90fbaae3e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [9]:
# 8. IMPORTANT: back this up to Drive immediately so you never redo steps 6-7 again
!mkdir -p {BACKUP_DIR}
!cp data/processed/*.csv data/processed/*.json {BACKUP_DIR}/
!cp data/raw/Reviews.csv {BACKUP_DIR}/
print('Backed up to Drive.')

^C
^C
Backed up to Drive.


## Everyone runs from here down

In [6]:
# 9. Exploratory data analysis on the real cleaned data
!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

Total reviews: 536837
Unique products: 43824

Missing values per column:
Id                         0
ProductId                  0
UserId                     0
ProfileName               25
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

Average review length: 79.9 words

Figures saved to outputs/figures


In [7]:
# 10. BART baseline test -- pretrained model, no fine-tuning yet.
# Proves the model implementation works (Milestone 3 requirement).
from transformers import BartForConditionalGeneration, BartTokenizer
import pandas as pd

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

df = pd.read_csv('data/processed/clean_reviews.csv')
sample_reviews = df['Text'].sample(3, random_state=1).tolist()

for i, review in enumerate(sample_reviews):
    inputs = tokenizer(review, return_tensors='pt', max_length=1024, truncation=True)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=60, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f'--- Review {i+1} ---')
    print('Original:', review[:200], '...')
    print('BART summary:', summary)
    print()

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

--- Review 1 ---
Original: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! ...
BART summary: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! will buy again. Will buy another 24 can package of Pedicure Dog Food for our dogs.

--- Review 2 ---
Original: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it ...
BART summary: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it hold me over past lunch. Love

--- Review 3 ---
Original: These bars are great for a snack. They have alot of healthy ingredients and low suga

In [9]:
# 11. Prompted LLM test -- uses Colab Secrets for the API key (key icon in left sidebar).
# Add a secret named ANTHROPIC_API_KEY, enable notebook access, then run this.
import os, sys, json
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, '.')
from models.llm_prompting import summarize_products

batches = json.load(open('data/processed/product_batches.json'))
results = summarize_products(batches[:3])  # just 3 products as a test

for r in results:
    print(r['product_id'])
    print(r.get('summary') or r.get('error'))
    print('---')

0006641040
Overall sentiment: Customers overwhelmingly love the classic content and nostalgic value of Maurice Sendak's "Chicken Soup with Rice," but many are disappointed by the small physical size of this particular edition.

Pros:
- Beloved, timeless content with catchy, rhythmic poems that children and adults adore across generations, making it an effective and fun way to teach months of the year.
- Strong nostalgic appeal that resonates with parents and grandparents who grew up with the book, creating meaningful shared reading experiences with their children.

Cons:
- The book is much smaller than expected (approximately 4x5 inches), leaving many buyers feeling it is undersized and poor value for the price.
- The physical quality of this edition feels cheap and underwhelming, making some buyers reluctant to give it as a gift.

Recommendation: Buy this book for its wonderful, classic content, but consider seeking out a larger used edition or a different printing if physical size an